In [1]:
import autogen
from agents import manager, user_proxy
extracted_phenotypes_path = "./results/Case1/aracrop_phenotypes.csv"

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

In [2]:
# Task 4: statistical test
# Rep 0: Save chat history
task = f'''
Given {extracted_phenotypes_path}, perform a mixed-design repeated-measures ANOVA statistical test, where:
    - Dependent variable: projected_leaf_area
    - Within-subject factor: days_after_sowing
    - Between-subject factor: ecotype
    - Subject identifier: plant_id
Following ANOVA, perform a Tukey–Kramer post-hoc test to identify which pairs of ecotypes significantly differ from each other on projected_leaf_area (i.e. PLA).
Save the ANOVA results to pla_anova.csv, and the post-hoc results to pla_tukey.csv in the directory ./results/Case1_Task4_wlog/.
'''
logging_session_id = autogen.runtime_logging.start(logger_type="file", config={"filename": "runtime_case1_task4.log"})
print("Logging session ID: " + str(logging_session_id))
res = user_proxy.initiate_chat(recipient=manager, message=task,)
autogen.runtime_logging.stop()

Logging session ID: 03bcf3ee-ca27-471b-84a8-6315ae27da3a
Admin (to manager):


Given ./results/Case1/aracrop_phenotypes.csv, perform a mixed-design repeated-measures ANOVA statistical test, where:
    - Dependent variable: projected_leaf_area
    - Within-subject factor: days_after_sowing
    - Between-subject factor: ecotype
    - Subject identifier: plant_id
Following ANOVA, perform a Tukey–Kramer post-hoc test to identify which pairs of ecotypes significantly differ from each other on projected_leaf_area (i.e. PLA).
Save the ANOVA results to pla_anova.csv, and the post-hoc results to pla_tukey.csv in the directory ./results/Case1_Task4_wlog/.


--------------------------------------------------------------------------------


manager (to Admin):

***** Suggested tool call (call_JqA70e4DaB2blTItdXuDDnMq): make_dir *****
Arguments: 
{"dir_path":"./results/Case1_Task4_wlog/"}
*************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION make_dir...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_JqA70e4DaB2blTItdXuDDnMq) *****
Directory './results/Case1_Task4_wlog/' already exists.
**********************************************************************

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_zQhPaatnfHz3525OyJ494jSh): perform_anova *****
Arguments: 
{"data_path": "./results/Case1/aracrop_phenotypes.csv", "descriptor": "projected_leaf_area", "within_subject_factor": "days_after_sowing", "between_subjec

In [6]:
import pandas as pd
# validate if the results of wlog = org
# org results (copy from case1.ipynb)
# >>>>>>>> EXECUTING FUNCTION perform_anova...
#               Source            SS  DF1  DF2           MS           F  \
# 0            ecotype  12523.073369    4   19  3130.768342   44.908296   
# 1  days_after_sowing  23588.800408   51  969   462.525498  221.942711   
# 2        Interaction   9636.465702  204  969    47.237577   22.666936   

#            p-unc       np2       eps  
# 0   1.988585e-09  0.904346       NaN  
# 1   0.000000e+00  0.921143  0.023693  
# 2  2.383402e-262  0.826749       NaN
 
# >>>>>>>> EXECUTING FUNCTION perform_tukey_test...
#       A     B   mean(A)   mean(B)      diff        se          T  \
# 0  adh1  col0  4.403086  7.563659 -3.160573  0.776724  -4.069107    
# 1  adh1   ctr  4.403086  0.386135  4.016951  0.776724   5.171658   
# 2  adh1  ein2  4.403086  8.948157 -4.545071  0.776724  -5.851590   
# 3  adh1   pgm  4.403086  2.869854  1.533231  0.776724   1.973972   
# 4  col0   ctr  7.563659  0.386135  7.177524  0.732302   9.801311   
# 5  col0  ein2  7.563659  8.948157 -1.384498  0.732302  -1.890609   
# 6  col0   pgm  7.563659  2.869854  4.693805  0.732302   6.409653   
# 7   ctr  ein2  0.386135  8.948157 -8.562022  0.732302 -11.691920   
# 8   ctr   pgm  0.386135  2.869854 -2.483720  0.732302  -3.391658
# 9  ein2   pgm  8.948157  2.869854  6.078302  0.732302   8.300262   

#         p-tukey    hedges  
# 0  5.220275e-03 -2.632246  
# 1  4.644033e-04  8.140057  
# 2  1.080956e-04 -2.530038  
# 3  3.152301e-01  2.399903  
# 4  6.727681e-08  7.092351  
# 5  3.553312e-01 -0.734539  
# 6  3.385964e-05  4.351188  
# 7  3.748997e-09 -5.375180  
# 8  2.257755e-02 -6.449929  
# 9  8.840395e-07  3.715140  

anova_org = [
    ["Source", "SS", "DF1", "DF2", "MS", "F", "p-unc", "np2", "eps"],
    ["ecotype", 12523.073369, 4, 19, 3130.768342, 44.908296, 1.988585e-09, 0.904346, None],
    ["days_after_sowing", 23588.800408, 51, 969, 462.525498, 221.942711, 0.000000e+00, 0.921143, 0.023693],
    ["Interaction", 9636.465702, 204, 969, 47.237577, 22.666936, 2.383402e-262, 0.826749, None],
]

tukey_org = [
        ["A", "B", "mean(A)", "mean(B)", "diff", "se", "T", "p-tukey", "hedges"],
        ["adh1",  "col0", 4.403086, 7.563659, -3.160573, 0.776724, -4.069107, 5.220275e-03, -2.632246],
        ["adh1",  "ctr",  4.403086,  0.386135,  4.016951,  0.776724,  5.171658,  4.644033e-04,  8.140057],
        ["adh1",  "ein2",  4.403086,  8.948157,  -4.545071,  0.776724,  -5.851590,  1.080956e-04,  -2.530038], 
        ["adh1",  "pgm",  4.403086,  2.869854,  1.533231,  0.776724,  1.973972,  3.152301e-01,  2.399903],
        ["col0",  "ctr",  7.563659,  0.386135,  7.177524,  0.732302,  9.801311,  6.727681e-08,  7.092351],
        ["col0",  "ein2",  7.563659,  8.948157,  -1.384498,  0.732302,  -1.890609,  3.553312e-01,  -0.734539],
        ["col0",  "pgm",  7.563659,  2.869854,  4.693805,  0.732302,  6.409653,  3.385964e-05,  4.351188],
        ["ctr",  "ein2",  0.386135,  8.948157,  -8.562022,  0.732302,  -11.691920,  3.748997e-09,  -5.375180],
        ["ctr",  "pgm",  0.386135,  2.869854,  -2.483720,  0.732302,  -3.391658,  2.257755e-02,  -6.449929],
        ["ein2", "pgm",  8.948157,  2.869854,  6.078302,  0.732302,  8.300262,  8.840395e-07,  3.715140],
]

# Convert to DataFrame
anova_csv = pd.DataFrame(anova_org[1:], columns=anova_org[0])
tukey_csv = pd.DataFrame(tukey_org[1:], columns=tukey_org[0])
anova_wlog = pd.read_csv('./results/Case1_Task4_wlog/pla_anova.csv')
tukey_wlog = pd.read_csv('./results/Case1_Task4_wlog/pla_tukey.csv')

def compare_csv_approx(df1, df2, decimals=3):
    if not df1.columns.equals(df2.columns):
        print("Columns differ.")
        return False
    if df1.shape != df2.shape:
        print("Shapes differ.")
        return False
    # Round numeric columns
    for col in df1.columns:
        if pd.api.types.is_numeric_dtype(df1[col]):
            df1[col] = df1[col].round(decimals)
            df2[col] = df2[col].round(decimals)

    # Compare all cells
    return df1.equals(df2)

print(compare_csv_approx(anova_csv, anova_wlog))
print(compare_csv_approx(tukey_csv, tukey_wlog))

True
True


In [1]:
# Task 4: statistical test
# Rep 1: saving executed pipeline
from agents import manager, user_proxy
task = f'''Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.'''
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_DKDbJCthoQ7ipBBzDlZ7GK9a): extract_pipeline *****
Arguments: 
{"pipeline_name":"ara_crop_stat","chat_log_path":"./autogen_logs/runtime_case1_task4.log"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION extract_pipeline...
Admin (to pipeline_summariser):

Extract a pipeline from the chat log. Name the extracted pipeline as ara_crop_stat. Users should be able to use new data to run the pipeline.
        Here is the chat_log [{'source_id': 140122037162832, 'source_n

In [1]:
# Task 4: statistical test
# Rep 2: saving executed pipeline
from agents import manager, user_proxy
rep = 2
task = f"Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_{rep}. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_2. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_WBKLSqfdr9Rngqh9qzBHPLs6): extract_pipeline *****
Arguments: 
{"pipeline_name":"ara_crop_stat_2","chat_log_path":"./autogen_logs/runtime_case1_task4.log"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION extract_pipeline...
Admin (to pipeline_summariser):

Extract a pipeline from the chat log. Name the extracted pipeline as ara_crop_stat_2. Users should be able to use new data to run the pipeline.
        Here is the chat_log [{'source_id': 140122037162832, 'so

In [1]:
# Task 4: statistical test
# Rep 3: saving executed pipeline
from agents import manager, user_proxy
rep = 3
task = f"Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_{rep}. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_3. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_kjzaTH5dxcxwaOn5683HNyLc): extract_pipeline *****
Arguments: 
{"pipeline_name":"ara_crop_stat_3","chat_log_path":"./autogen_logs/runtime_case1_task4.log"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION extract_pipeline...
Admin (to pipeline_summariser):

Extract a pipeline from the chat log. Name the extracted pipeline as ara_crop_stat_3. Users should be able to use new data to run the pipeline.
        Here is the chat_log [{'source_id': 140122037162832, 'so

In [1]:
# Task 4: statistical test
# Rep 4: saving executed pipeline
from agents import manager, user_proxy
rep = 4
task = f"Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_{rep}. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_4. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_hV8zfBGkkgZW06TIMViJcy2D): extract_pipeline *****
Arguments: 
{"pipeline_name":"ara_crop_stat_4","chat_log_path":"./autogen_logs/runtime_case1_task4.log"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION extract_pipeline...
Admin (to pipeline_summariser):

Extract a pipeline from the chat log. Name the extracted pipeline as ara_crop_stat_4. Users should be able to use new data to run the pipeline.
        Here is the chat_log [{'source_id': 140122037162832, 'so

In [1]:
# Task 4: statistical test
# Rep 5: saving executed pipeline
from agents import manager, user_proxy
rep = 5
task = f"Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_{rep}. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Summarise the executed function calls and code into a reproducible pipeline. Name it as ara_crop_stat_5. The chatlog was saved at ./autogen_logs/runtime_case1_task4.log.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_fA63lDX8Rx8LSUGvZUW7cMVy): extract_pipeline *****
Arguments: 
{"pipeline_name":"ara_crop_stat_5","chat_log_path":"./autogen_logs/runtime_case1_task4.log"}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION extract_pipeline...
Admin (to pipeline_summariser):

Extract a pipeline from the chat log. Name the extracted pipeline as ara_crop_stat_5. Users should be able to use new data to run the pipeline.
        Here is the chat_log [{'source_id': 140122037162832, 'so

In [1]:
# Execution rep 1
from agents import manager, user_proxy
extracted_phenotypes_path_for_pipe = "./results/Case1/aracrop_phenotypes_pipe_repro_input.csv"
results_dir = "./results/Case1_Task4_wlog"
task = f"Run the ara_crop_stat pipeline with {extracted_phenotypes_path_for_pipe}, change output_dir to {results_dir}_for_pipe."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Run the ara_crop_stat pipeline with ./results/Case1/aracrop_phenotypes_pipe_repro_input.csv, change output_dir to ./results/Case1_Task4_wlog_for_pipe.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_98ac0oCqsrWmfMLHLu1u54IG): get_pipeline_zoo *****
Arguments: 
{}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_pipeline_zoo...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_98ac0oCqsrWmfMLHLu1u54IG) *****
{"ara_crop_pipeline": {"function_name": "ara_crop_pipeline", "description": "Pipeline to compute phenotypes for Arabidopsis plant images, merge with metadata, and save the results.\nArgs:\n    metadata_path (str): Path to the metadata JS

In [1]:
# Execution rep 2
rep = 2
from agents import manager, user_proxy
extracted_phenotypes_path_for_pipe = "./results/Case1/aracrop_phenotypes_pipe_repro_input.csv"
results_dir = "./results/Case1_Task4_wlog"
task = f"Run the ara_crop_stat_{rep} pipeline with {extracted_phenotypes_path_for_pipe}, change output_dir to {results_dir}_for_pipe_{rep}."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Run the ara_crop_stat_2 pipeline with ./results/Case1/aracrop_phenotypes_pipe_repro_input.csv, change output_dir to ./results/Case1_Task4_wlog_for_pipe_2.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_UaqvCRoqo3HThJxiqL1unAEp): get_pipeline_zoo *****
Arguments: 
{}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_pipeline_zoo...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_UaqvCRoqo3HThJxiqL1unAEp) *****
{"ara_crop_pipeline": {"function_name": "ara_crop_pipeline", "description": "Pipeline to compute phenotypes for Arabidopsis plant images, merge with metadata, and save the results.\nArgs:\n    metadata_path (str): Path to the metadat

In [1]:
# Execution rep 3
rep = 3
from agents import manager, user_proxy
extracted_phenotypes_path_for_pipe = "./results/Case1/aracrop_phenotypes_pipe_repro_input.csv"
results_dir = "./results/Case1_Task4_wlog"
task = f"Run the ara_crop_stat_{rep} pipeline with {extracted_phenotypes_path_for_pipe}, change output_dir to {results_dir}_for_pipe_{rep}."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Run the ara_crop_stat_3 pipeline with ./results/Case1/aracrop_phenotypes_pipe_repro_input.csv, change output_dir to ./results/Case1_Task4_wlog_for_pipe_3.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_a9uNgruJOJckISSNufczCRza): get_pipeline_zoo *****
Arguments: 
{}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_pipeline_zoo...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_a9uNgruJOJckISSNufczCRza) *****
{"ara_crop_pipeline": {"function_name": "ara_crop_pipeline", "description": "Pipeline to compute phenotypes for Arabidopsis plant images, merge with metadata, and save the results.\nArgs:\n    metadata_path (str): Path to the metadat

In [1]:
# Execution rep 4
rep = 4
from agents import manager, user_proxy
extracted_phenotypes_path_for_pipe = "./results/Case1/aracrop_phenotypes_pipe_repro_input.csv"
results_dir = "./results/Case1_Task4_wlog"
task = f"Run the ara_crop_stat_{rep} pipeline with {extracted_phenotypes_path_for_pipe}, change output_dir to {results_dir}_for_pipe_{rep}."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Run the ara_crop_stat_4 pipeline with ./results/Case1/aracrop_phenotypes_pipe_repro_input.csv, change output_dir to ./results/Case1_Task4_wlog_for_pipe_4.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_2VE5V1sDsI26biGqrwKKe4TD): get_pipeline_zoo *****
Arguments: 
{}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_pipeline_zoo...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_2VE5V1sDsI26biGqrwKKe4TD) *****
{"ara_crop_pipeline": {"function_name": "ara_crop_pipeline", "description": "Pipeline to compute phenotypes for Arabidopsis plant images, merge with metadata, and save the results.\nArgs:\n    metadata_path (str): Path to the metadat

In [1]:
# Execution rep 5
rep = 5
from agents import manager, user_proxy
extracted_phenotypes_path_for_pipe = "./results/Case1/aracrop_phenotypes_pipe_repro_input.csv"
results_dir = "./results/Case1_Task4_wlog"
task = f"Run the ara_crop_stat_{rep} pipeline with {extracted_phenotypes_path_for_pipe}, change output_dir to {results_dir}_for_pipe_{rep}."
res = user_proxy.initiate_chat(recipient=manager, message=task,)

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

Admin (to manager):

Run the ara_crop_stat_5 pipeline with ./results/Case1/aracrop_phenotypes_pipe_repro_input.csv, change output_dir to ./results/Case1_Task4_wlog_for_pipe_5.

--------------------------------------------------------------------------------
manager (to Admin):

***** Suggested tool call (call_HGaTqgZOPWCN8A0pEPOQO2kK): get_pipeline_zoo *****
Arguments: 
{}
*********************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> NO HUMAN INPUT RECEIVED.

>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_pipeline_zoo...
Admin (to manager):

Admin (to manager):

***** Response from calling tool (call_HGaTqgZOPWCN8A0pEPOQO2kK) *****
{"ara_crop_pipeline": {"function_name": "ara_crop_pipeline", "description": "Pipeline to compute phenotypes for Arabidopsis plant images, merge with metadata, and save the results.\nArgs:\n    metadata_path (str): Path to the metadat

In [1]:
# validate results
import pandas as pd
anova_wlog = pd.read_csv('./results/Case1_Task4_wlog/pla_anova.csv')
tukey_wlog = pd.read_csv('./results/Case1_Task4_wlog/pla_tukey.csv')

anova_rep1 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe/pla_anova.csv')
tukey_rep1 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe/pla_tukey.csv')
print(anova_wlog.equals(anova_rep1))
print(tukey_wlog.equals(tukey_rep1))

anova_rep2 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_2/pla_anova.csv')
tukey_rep2 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_2/pla_tukey.csv')
print(anova_wlog.equals(anova_rep2))
print(tukey_wlog.equals(tukey_rep2))

anova_rep3 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_3/anova_results.csv')
tukey_rep3 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_3/tukey_results.csv')
print(anova_wlog.equals(anova_rep3))
print(tukey_wlog.equals(tukey_rep3))

anova_rep4 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_4/anova_results.csv')
tukey_rep4 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_4/tukey_results.csv')
print(anova_wlog.equals(anova_rep4))
print(tukey_wlog.equals(tukey_rep4))

anova_rep5 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_5/pla_anova.csv')
tukey_rep5 = pd.read_csv('./results/Case1_Task4_wlog_for_pipe_5/pla_tukey.csv')
print(anova_wlog.equals(anova_rep5))
print(tukey_wlog.equals(tukey_rep5))

True
True
True
True
True
True
True
True
True
True
